In [2]:
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

project = Path.cwd().parent

features = pd.read_csv(
    project / "data" / "processed" / "customer_features.csv"
)

features["SIGNUP_DATE"] = pd.to_datetime(features["SIGNUP_DATE"])
features = features.sort_values("SIGNUP_DATE").reset_index(drop=True)

cutoff_date = features["SIGNUP_DATE"].quantile(0.8)

train_mask = features["SIGNUP_DATE"] <= cutoff_date
test_mask = features["SIGNUP_DATE"] > cutoff_date

y = features["CHURNED"].astype(int)

demographic_features = [
    "AGE",
    "GENDER",
    "CITY",
    "MEMBERSHIP_TIER",
    "MARKETING_OPT_IN",
    "PREFERRED_CATEGORY",
    "tenure_months"
]

full_features = [
    col for col in features.columns
    if col not in ["CUSTOMER_ID", "SIGNUP_DATE", "CHURNED"]
]

models = {
    "Demographics Only": demographic_features,
    "Behavioural + Engagement + Support": full_features
}

results = []

for name, columns in models.items():
    X = features[columns].copy()

    categorical_columns = X.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()

    numeric_columns = X.select_dtypes(
        exclude=["object", "string"]
    ).columns.tolist()

    preprocessor = ColumnTransformer([
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_columns
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_columns
        )
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])

    model.fit(
        X.loc[train_mask],
        y.loc[train_mask]
    )

    probability = model.predict_proba(
        X.loc[test_mask]
    )[:, 1]

    prediction = (probability >= 0.70).astype(int)

    results.append({
        "Feature Set": name,
        "Precision": round(
            precision_score(y.loc[test_mask], prediction, zero_division=0), 4
        ),
        "Recall": round(
            recall_score(y.loc[test_mask], prediction, zero_division=0), 4
        ),
        "F1": round(
            f1_score(y.loc[test_mask], prediction, zero_division=0), 4
        ),
        "PR-AUC": round(
            average_precision_score(y.loc[test_mask], probability), 4
        )
    })

ablation_results = pd.DataFrame(results)

ablation_results

,Feature Set,Precision,Recall,F1,PR-AUC
0,Demographics Only,0.0,0.0000,0.0000,0.0965
1,Behavioural + Engagement + Support,1.0,0.5106,0.6761,0.8271
